# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

> **Note**: All references are provided via their canonical `@id` fields, consistent with the Croissant schema.

In [ ]:
# List all record sets, their @ids, and their schema
print("Available RecordSets and fields in the dataset:")

if not hasattr(metadata, 'record_sets'):
    # Fallback: Some Croissant schemas use `recordSet` per the '@context'
    record_sets = getattr(metadata, 'record_set', [])
else:
    record_sets = metadata.record_sets

for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}")
    fields = rs.get('fields', [])
    if not fields and 'field' in rs:
        if isinstance(rs['field'], list):
            fields = rs['field']
        else:
            fields = [rs['field']]
    print("  Fields (by @id):")
    for field in fields:
        if isinstance(field, dict):
            print(f"    - {field['@id']} (label: {field.get('name', field.get('@id', ''))})")
        else:
            print(f"    - {field}")
    print()
if not record_sets:
    print("No record sets were found in the top-level metadata. Attempting to introspect dataset...")
    # mlcroissant infers record_set ids from the schema
    # Try fetching available record set ids via dataset.records.__signature__
    # As a workaround, we fetch from the schema directly
    if hasattr(metadata, '_jsonld') or hasattr(metadata, 'to_json'):
        json_dict = metadata._jsonld if hasattr(metadata, '_jsonld') else metadata.to_json()
        # Look for 'recordSet' or 'cr:recordSet' keys
        if 'recordSet' in json_dict:
            record_sets = json_dict['recordSet']
        elif 'cr:recordSet' in json_dict:
            record_sets = json_dict['cr:recordSet']
        else:
            record_sets = []
        if isinstance(record_sets, dict):
            record_sets = [record_sets]
        for rs in record_sets:
            print(f"- RecordSet @id: {rs['@id']}")
            fields = rs.get('fields', [])
            if not fields and 'field' in rs:
                if isinstance(rs['field'], list):
                    fields = rs['field']
                else:
                    fields = [rs['field']]
            print("  Fields (by @id):")
            for field in fields:
                if isinstance(field, dict):
                    print(f"    - {field['@id']} (label: {field.get('name', field.get('@id', ''))})")
                else:
                    print(f"    - {field}")
            print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

_Below, we dynamically extract all record sets, load their records into DataFrames keyed by their `@id`._

In [ ]:
# Helper to extract record set ids using Croissant conventions
def get_recordset_ids(metadata):
    rs_ids = []
    try:
        if hasattr(metadata, 'record_sets'):
            rs_list = metadata.record_sets
        elif hasattr(metadata, 'record_set'):
            rs_list = metadata.record_set
        else:
            # fallback: get from _jsonld if present
            json_dict = metadata._jsonld if hasattr(metadata, '_jsonld') else metadata.to_json()
            rs_list = json_dict.get('recordSet', json_dict.get('cr:recordSet', []))
            if isinstance(rs_list, dict):
                rs_list = [rs_list]
        for rs in rs_list:
            rs_ids.append(rs['@id'])
    except Exception as e:
        print(f"Could not extract record sets: {e}")
    return rs_ids

record_sets_ids = get_recordset_ids(metadata)

dataframes = {}
print('Extracting available record sets as DataFrames:')
for record_set_id in record_sets_ids:
    print(f"Loading record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"- Columns in {record_set_id}: {list(df.columns)}\n")

# For demonstration, let's pick the first available record set for further EDA/visualization.
if record_sets_ids:
    chosen_record_set_id = record_sets_ids[0]
    print(f"Defaulting to first RecordSet: {chosen_record_set_id}")
else:
    chosen_record_set_id = None
display_columns = dataframes[chosen_record_set_id].columns.tolist() if chosen_record_set_id else []
print('Available columns:', display_columns)

if chosen_record_set_id:
    dataframes[chosen_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# EDA on a selected numeric field
# Find the first column with numeric dtype to use for analysis
numeric_field = None
if chosen_record_set_id:
    df = dataframes[chosen_record_set_id]
    for col in df.columns:
        # Try to convert to numeric to guess
        sample = pd.to_numeric(df[col], errors='coerce')
        if sample.notnull().sum() > 0:
            numeric_field = col
            break

print(f"Selected field for numeric analysis: {numeric_field}")

if numeric_field is not None:
    # Clean/convert
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].mean() if df[numeric_field].notnull().sum() > 0 else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to group by a likely categorical column (e.g., first non-numeric, non-id column)
    group_field = None
    for col in df.columns:
        if col != numeric_field and not col.lower().startswith('id'):
            # Try to avoid grouping by numeric fields
            if not np.issubdtype(df[col].dtype, np.number):
                group_field = col
                break
    print(f"Grouping field chosen: {group_field}")
    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean of {numeric_field}):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Plotting data distributions
if chosen_record_set_id and numeric_field is not None:
    df = dataframes[chosen_record_set_id]
    plt.figure(figsize=(10, 5))
    df[numeric_field].hist(bins=30)
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.title(f'Histogram of {numeric_field} in {chosen_record_set_id}')
    plt.show()

    # If grouped, show bar chart
    if 'grouped_df' in locals() and group_field is not None:
        grouped_df_sorted = grouped_df.sort_values(by=numeric_field, ascending=False).head(10)
        plt.figure(figsize=(10, 5))
        plt.bar(grouped_df_sorted[group_field], grouped_df_sorted[numeric_field])
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.title(f"Top 10 {group_field} by mean {numeric_field}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* The dataset is successfully loaded using `mlcroissant` from the Croissant schema at the provided URL.
* Record sets, fields, and entity relationships are accessed via their canonical `@id` fields, ensuring precise referencing.
* Simple EDA and visualization demonstrate how to process and explore the dataset. For more advanced analysis, consider exploring additional record sets, fields, and annotations provided in the Croissant schema.
* See the [mlcroissant documentation](https://github.com/mlcommons/croissant) for more advanced integration and analysis examples.